# Unified post-training analysis notebook — dpo-safety-representations

One notebook for the whole analysis pipeline, orchestrating repository scripts (not duplicating
their logic). Components 1–4 (behavioral eval, activation extraction, probes, refusal direction —
plus direction-stability, bootstrap, bottleneck-layer, and now cross-branch comparison) each
internally loop over their own fixed `STAGES` list — now all 9 stages: `M0, M1, M2, M3, M3_direct,
M1_alt, M2_alt, M3_alt, M3_direct_alt`. All are resumable AND tolerant of partial readiness — the
alt branch trains and pushes independently across sessions, so at any given time some stages may
not exist yet. Missing stages are skipped with a clear message, not a crash; already-computed
results are never recomputed/overwritten. Component 5 (causal ablation) and 5b (steering) are
per-stage, GPU-heavy, and genuinely one-run-at-a-time — pick the stage in the Configuration cell.

Toggle `COMPONENTS_TO_RUN` below to run only what you need (e.g. while debugging one component).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/urosavurdic/dpo-safety-representations.git"
REPO_DIR = "/content/dpo-safety-representations"
BRANCH = "main"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y torchao

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
!pytest tests/ -v

## Verify prerequisites

Components 2–5b all build on each other (activations need the eval set; refusal direction needs
activations; causal ablation and steering need a direction). This just tells you up front what's
missing rather than failing halfway through a long run.

In [ ]:
required_for = {
    "controlled eval set": "data/processed/controlled_eval.jsonl",
}
for name, path in required_for.items():
    print(f"{'OK' if os.path.exists(path) else 'MISSING':7s} {name}: {path}")

## Configuration

`COMPONENTS_TO_RUN`: which analysis stages to execute this pass. `STAGE_FOR_CAUSAL`: which single
model to run causal ablation / steering against (GPU-heavy, one at a time) — must have activations
and a refusal direction already (Components 2 and 4).

In [ ]:
# ---- EDIT THIS ----
COMPONENTS_TO_RUN = {
    "behavioral": True,
    "activations": True,
    "probes": True,
    "refusal_direction": True,
    "causal_ablation": True,
    "steering": False,   # GPU-heavy, run separately once you know what layers/alpha you want
}
# All 8 stages have adapters on HF (M0 needs none) - Components 1-4 above process ALL of
# STAGES automatically in one pass, nothing to configure per-stage for those. Components 5/5b
# below load one full model at a time (GPU constraint) - list every stage you want them to run
# for and the cells loop over it automatically. The 4 real DPO endpoints are the ones causal
# ablation/steering actually test (M1/M2/M1_alt/M2_alt are SFT-only, no DPO effect to ablate).
STAGES_FOR_CAUSAL = ["M3", "M3_direct", "M3_alt", "M3_direct_alt"]
# --------------------

print("Will run:", [k for k, v in COMPONENTS_TO_RUN.items() if v])
print("Causal-ablation/steering target stages:", STAGES_FOR_CAUSAL)

## Component 1 — Behavioral evaluation

All stages, resumable. Wilson 95% CIs applied automatically by `reclassify_behavioral.py`.

In [ ]:
if COMPONENTS_TO_RUN["behavioral"]:
    !python -m src.analysis.eval_behavioral
    !python -m src.analysis.reclassify_behavioral

## Component 2 — Activation extraction

GPU. All stages, resumable per-stage.

In [ ]:
if COMPONENTS_TO_RUN["activations"]:
    !python -m src.analysis.eval_extract_activations

## Component 3 — Linear probes

CPU, needs Component 2's activations. Held-out flagging rate, not the retired CV-accuracy metric.

In [ ]:
if COMPONENTS_TO_RUN["probes"]:
    !python -m src.analysis.eval_probes
    !python -m src.analysis.summarize_probe_findings

## Component 4 — Refusal direction, stability, bootstrap, bottleneck layer, cross-branch

CPU, needs Component 2's activations. `direction_stability`/`bootstrap_direction_stability` read
`cosine_similarity.json` produced by `eval_refusal_direction` in this same cell, so order matters
here specifically. `eval_refusal_direction` now also computes `cross_branch` comparisons
(M1 vs M1_alt, M2 vs M2_alt, M3 vs M3_alt, M3_direct vs M3_direct_alt) whenever both sides of a
pair have activations — this is the core answer to "is this finding dataset-specific."
`summarize_cross_branch` pulls that together with behavioral and probe results into one
side-by-side comparison, printed and saved to `results/summaries/cross_branch_comparison.json`.

In [ ]:
if COMPONENTS_TO_RUN["refusal_direction"]:
    !python -m src.analysis.eval_refusal_direction
    !python -m src.interpretability.direction_stability
    !python -m src.interpretability.bootstrap_direction_stability
    !python -m src.interpretability.bottleneck_layer
    !python -m src.analysis.summarize_cross_branch

## Component 5 — Causal ablation

GPU. Runs against `STAGE_FOR_CAUSAL` (narrow layer range 24–28, the range Component 4/ablation
already validated as sufficient — see `ABLATE_LAYERS` in `eval_causal_ablation.py` if you want the
wider 14–28 range instead). Then Wilson-CI summary, paired McNemar test, and a bootstrap CI on the
effect size — all reused, none reimplemented here.

In [ ]:
if COMPONENTS_TO_RUN["causal_ablation"]:
    for stage in STAGES_FOR_CAUSAL:
        print(f"\n{'='*20} Causal ablation: {stage} {'='*20}")
        !python -m src.analysis.eval_causal_ablation --stage {stage}

        suffix = "narrow"  # matches the current ABLATE_LAYERS default (24-28)
        raw_file = (f"results/raw/causal_ablation_raw_{suffix}.json" if stage == "M3"
                    else f"results/raw/causal_ablation_raw_{stage.lower()}_{suffix}.json")

        !python -m src.analysis.summarize_causal_ablation --file {raw_file} --stage {stage}
        !python -m src.analysis.mcnemar_causal_ablation --file {raw_file} --conditions {stage}_baseline {stage}_ablated
        !python -m src.analysis.bootstrap_causal_effect --file {raw_file} --quadrant C --category soft_deflection
        !python -m src.analysis.bootstrap_causal_effect --file {raw_file} --quadrant A --category refusal

## Component 5b — Steering (optional, run separately)

GPU. Uses `eval_steering_v2.py` — configurable layers/alpha/quadrants, never overwrites a previous
run. Defaults below: single layer 24 (inside the ablation-validated range, unlike the original
notebook's L21), natural per-layer alpha, testing both quadrant D (over-refusal) and quadrant A
(side-effect check) in one run. Edit the flags directly for a different layer/alpha/quadrant
combination — see `eval_steering_v2.py`'s docstring for the full option set.

In [ ]:
if COMPONENTS_TO_RUN["steering"]:
    for stage in STAGES_FOR_CAUSAL:
        print(f"\n{'='*20} Steering: {stage} {'='*20}")
        !python -m src.analysis.eval_steering_v2 --stage {stage} --layers 24 --quadrants A D
        # Each run prints the exact summarize/mcnemar follow-up commands for its own output file.

## Results summary — what's on disk now

In [ ]:
import subprocess
for d in ["results/behavioral_eval", "results/probes", "results/refusal_direction",
          "results/interpretability", "results/raw", "results/summaries"]:
    if os.path.exists(d):
        print(f"\n{d}/")
        for f in sorted(os.listdir(d)):
            print(f"  {f}")